Transform Customer Data   

In [0]:
select * from ecomm.bronze.v_customers


#Remove records with null customer_i

In [0]:
%sql
select * 
from ecomm.bronze.v_customers
where customer_id is not null

In [0]:
%sql
select * 
from ecomm.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
select distinct *
from ecomm.bronze.v_customers
where customer_id is not null
order by customer_id   

#common Table expression (CTE)

common Table expression (CTE) is a temporary result set that is defined within the execution scope of a single SQL statement. A CTE can be thought of as a temporary view that is only visible to the current statement. A CTE can be used to:

- Simplify a complex query by breaking it up into smaller, more manageable parts.
- Remove repetitive code from a query.
- Make a query easier to read and write by giving it a name.
- Create a temporary view that can be used in subsequent queries.
- Create a temporary table that can be used in subsequent queries.

#comment CTAS (create tablet as select)

In [0]:
create or replace temporary view v_customers_distinct 
as
select distinct *
from ecomm.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
select *
from v_customers_distinct

In [0]:
select customer_id,
max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id

what is cte_max

In [0]:
with cte_max as (
    select customer_id,
max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id
)
select cd.* 
from v_customers_distinct cd
join cte_max cm
on cd.customer_id = cm.customer_id
and cd.created_timestamp = cm.max_created_timestamp

In [0]:
with cte_max as(
Select customer_id,
max(created_timestamp) as max_created_timestamps
from v_customers_distinct
group by customer_id
)
select cd.customer_id,
cast(cd.created_timestamp as timestamp) as created_timestamp,
cd.customer_name,
cast(cd.date_of_birth as date) as date_of_birth,
cd.email,
cast(cd.member_since as date) as member_since,
cd.telephone
from v_customers_distinct cd
join cte_max cm
on cd.customer_id = cm.customer_id
and cd.created_timestamp = cm.max_created_timestamp

In [0]:
create table ecomm.bronze.customers
as
with cte_max as(
Select customer_id,
max(created_timestamp) as max_created_timestamps
from v_customers_distinct
group by customer_id
)
select cd.customer_id,
cast(cd.created_timestamp as timestamp) as created_timestamp,
cd.customer_name,
cast(cd.date_of_birth as date) as date_of_birth,
cd.email,
cast(cd.member_since as date) as member_since,
cd.telephone
from v_customers_distinct cd
join cte_max cm
on cd.customer_id = cm.customer_id
and cd.created_timestamp = cm.max_created_timestamps

In [0]:
drop table ecomm.bronze.customers

with cte_max as(
Select customer_id,
max(created_timestamp) as max_created_timestamps
from v_customers_distinct
group by customer_id
)
select cd.customer_id,
cast(cd.created_timestamp as timestamp) as created_timestamp,
cd.customer_name,
cast(cd.date_of_birth as date) as date_of_birth,
cd.email,
cast(cd.member_since as date) as member_since,
cd.telephone
from v_customers_distinct cd
join cte_max cm
on cd.customer_id = cm.customer_id
and cd.created_timestamp = cm.max_created_timestamps

In [0]:
create table ecomm.silver.customers
as
with cte_max as(
Select customer_id,
max(created_timestamp) as max_created_timestamps
from v_customers_distinct
group by customer_id
)
select cd.customer_id,
cast(cd.created_timestamp as timestamp) as created_timestamp,
cd.customer_name,
cast(cd.date_of_birth as date) as date_of_birth,
cd.email,
cast(cd.member_since as date) as member_since,
cd.telephone
from v_customers_distinct cd
join cte_max cm
on cd.customer_id = cm.customer_id
and cd.created_timestamp = cm.max_created_timestamps

In [0]:
    select payment_id,
    order_id,
    payment_timestamp,
    payment_method,
    payment_status
    from
    ecomm.bronze.payments

In [0]:
select payment_id,
order_id,
cast(data_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
date_format(payment_timestamp, 'HH-MM-SS') as payment_time,
payment_method,
payment_status
from
ecomm.bronze.payments

In [0]:
select payment_id,
order_id,
cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
date_format(payment_timestamp, 'HH-MM-SS') as payment_time,
payment_status,
case payment_status 
when 1 then 'processing'
when 2 then 'approved'
when 3 then 'declined' 
when 4 then 'pending_for_bank'
end as  payment_status_described,
payment_method
from ecomm.bronze.payments

In [0]:
create table ecomm.silver.payments
as select payment_id,
order_id,
cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
date_format(payment_timestamp, 'HH-MM-SS') as payment_time,
payment_status,
case payment_status 
when 1 then 'processing'
when 2 then 'approved'
when 3 then 'declined' 
when 4 then 'pending_for_bank'
end as  payment_status_described,
payment_method
from ecomm.bronze.payments

Transform Address data

In [0]:
select * from ecomm.bronze.v_addresses